<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/HighFlyersProgram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install --upgrade yfinance
!pip install  --upgrade pandas_ta
!pip install ta pandas_ta
!pip install scipy==1.16.2
#!pip install numpy==1.26.4 scipy==1.11.4
#--force-reinstall --no-cache-dir

  Using cached ta-0.11.0.tar.gz (25 kB)
  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=80ff909b50b165705e87f257d3b42c2a3610a6be34bd19d6f0501c1f658955fa
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
Successfully built ta
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 24.7 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


In [3]:
import yfinance as yf
print(yf.__version__)
import pandas as pd
import numpy as np
from datetime import datetime
import time
import ta
import random
import requests
from datetime import datetime, timedelta
from scipy.stats import linregress
from transformers import pipeline


from dataclasses import dataclass, field
from typing import Optional
import warnings
warnings.filterwarnings("ignore")
# ensure reproducibility
random.seed(42)
print("Libraries Installed!")

1.3.0
Libraries Installed!


In [5]:
def get_last_quad_witching(reference_date=None):
    """
    Returns the most recent quad witching date (3rd Friday of Mar/Jun/Sep/Dec)
    before or equal to reference_date.
    """
    import calendar
    from datetime import date, timedelta

    if reference_date is None:
        reference_date = date.today()
    elif isinstance(reference_date, str):
        reference_date = pd.to_datetime(reference_date).date()

    quad_months = [3, 6, 9, 12]

    def third_friday(year, month):
        # Find first day of month
        first_day = date(year, month, 1)
        # Find first Friday
        first_friday = first_day + timedelta(days=(4 - first_day.weekday()) % 7)
        # Third Friday = first Friday + 14 days
        return first_friday + timedelta(days=14)

    # Generate last 2 years of quad witching dates
    candidates = []
    for year in [reference_date.year - 1, reference_date.year]:
        for month in quad_months:
            candidates.append(third_friday(year, month))

    # Filter to dates on or before reference_date
    past_dates = [d for d in candidates if d <= reference_date]

    # Return most recent
    return max(past_dates).strftime("%Y-%m-%d")

In [6]:
quad_witching_date = get_last_quad_witching()  # auto-detects most recent
print(quad_witching_date)

2026-03-20


In [7]:
today = datetime.today()
start_of_year = today.replace(month=1, day=1)

# First day of this month
first_day_month = today.replace(day=1)

# First day of this week (Monday as weekday 0)
first_day_week = today - timedelta(days=today.weekday())
print("First day of year:", start_of_year)

print("\nFirst day of this month:", first_day_month)
print("\nFirst day of this week:", first_day_week)

def most_recent_quarter_start(today=None):
    if today is None:
        today = pd.Timestamp.today().normalize()
    year = today.year
    month = today.month

    # Determine quarter start months: Jan, Apr, Jul, Oct
    if month >= 10:
        q_start = pd.Timestamp(year, 10, 1)
    elif month >= 7:
        q_start = pd.Timestamp(year, 7, 1)
    elif month >= 4:
        q_start = pd.Timestamp(year, 4, 1)
    else:
        q_start = pd.Timestamp(year, 1, 1)

    return q_start

# Example usage
print("Today:", pd.Timestamp.today().normalize())
print("Most recent quarter start:", most_recent_quarter_start())


First day of year: 2026-01-01 16:19:53.765920

First day of this month: 2026-05-01 16:19:53.765920

First day of this week: 2026-05-11 16:19:53.765920
Today: 2026-05-17 00:00:00
Most recent quarter start: 2026-04-01 00:00:00


In [10]:
# List of ETFs to analyze
#df_o = pd.read_csv('stock_list.csv')

df_raw = pd.read_csv('small_list.csv')
df_raw = df_raw[df_raw['Type'].isin(['Stock'])]
recent_quarter = most_recent_quarter_start()
etfs = df_raw['Asset'].to_list()
print(etfs)

print(len(etfs))

['RXT', 'MXL', 'INOD', 'AMBQ', 'DOCN', 'BAND', 'AAOI', 'VPG', 'PENG', 'SATL', 'BLZE', 'LUNR', 'XMTR', 'SITM', 'AEHR', 'MEI', 'WYFI', 'DMRC', 'SVCO', 'NVTS', 'HUT', 'VSH', 'PLUG', 'AIP', 'SYNA', 'KOPN', 'BKSY', 'CEVA', 'APLD', 'VECO', 'SMTC', 'CLFD', 'CXDO', 'OUST', 'FLY', 'DDD', 'RIOT', 'VICR', 'PL', 'AEVA', 'SEZL', 'BE', 'PDFS', 'VIAV', 'TTMI', 'RDW', 'BTDR', 'IONQ', 'NXDR', 'XPER', 'VOYG', 'CORZ', 'RMBS', 'FA', 'VSAT', 'KN', 'SHLS', 'LASR', 'AMBA', 'NAVN', 'FEIM', 'CRDO', 'NTCT', 'SANM', 'DGII', 'NVEC', 'MTRN', 'COHU', 'CIFR', 'RUM', 'NXT', 'SEI', 'RELL', 'PLAB', 'BHE', 'UIS', 'NOVT', 'HLIT', 'EXTR', 'MRX', 'NN', 'WULF', 'MPTI', 'OPLN', 'EGHT', 'ATEN', 'CRSR', 'WBTN', 'MARA', 'NSIT', 'DCO', 'GRND', 'MKTW', 'FN', 'PKE', 'TEAD']
96


## Filter for liquidity

In [11]:

# Filter ETFs or stocks for liquidity
def filter_by_liquidity(etf_df, ticker_col="Asset", min_dollar_vol=10e6, lookback_days=30):
    liquid_etfs = []

    for ticker in etf_df[ticker_col]:
        try:
            # Fetch daily historical data
            data = yf.download(ticker, period=f"{lookback_days*2}d", interval="1d", auto_adjust=True)

            if data.empty:
                continue

            # Calculate dollar volume (Close × Volume)
            data["dollar_volume"] = data["Close"] * data["Volume"]

            # Calculate rolling average over lookback_days
            avg_dollar_volume = data["dollar_volume"].rolling(window=lookback_days).mean().iloc[-1]

            # Check liquidity condition
            if avg_dollar_volume >= min_dollar_vol:
                liquid_etfs.append(ticker)

        except Exception as e:
            print(f"Error fetching {ticker}: {e}")

    # Return filtered DataFrame
    return etf_df[etf_df[ticker_col].isin(liquid_etfs)]

# Example usage
df = pd.DataFrame({"Assets": etfs})
liquid_df = filter_by_liquidity(df, ticker_col="Assets")
df_o = df_raw[df_raw['Asset'].isin(liquid_df['Assets'])]
etfs = df_o['Asset'].to_list()
print("")
print(etfs)
print(len(etfs))



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********


['RXT', 'MXL', 'INOD', 'AMBQ', 'DOCN', 'BAND', 'AAOI', 'VPG', 'PENG', 'SATL', 'BLZE', 'LUNR', 'XMTR', 'SITM', 'AEHR', 'MEI', 'WYFI', 'NVTS', 'HUT', 'VSH', 'PLUG', 'AIP', 'SYNA', 'KOPN', 'BKSY', 'CEVA', 'APLD', 'VECO', 'SMTC', 'CLFD', 'OUST', 'FLY', 'RIOT', 'VICR', 'PL', 'AEVA', 'SEZL', 'BE', 'PDFS', 'VIAV', 'TTMI', 'RDW', 'BTDR', 'IONQ', 'VOYG', 'CORZ', 'RMBS', 'FA', 'VSAT', 'KN', 'SHLS', 'LASR', 'AMBA', 'NAVN', 'CRDO', 'NTCT', 'SANM', 'DGII', 'MTRN', 'COHU', 'CIFR', 'RUM', 'NXT', 'SEI', 'PLAB', 'BHE', 'NOVT', 'HLIT', 'EXTR', 'MRX', 'NN', 'WULF', 'OPLN', 'ATEN', 'MARA', 'NSIT', 'DCO', 'GRND', 'FN']
79


# Classify Sector Stages ( Sam Weinstien Weekly Timeframe)

In [12]:

def weinstein_stage(df, sma_window=30,smaSlope_window=10):
    """Determine Weinstein stage using 30-week SMA and its slope."""
     # Handle MultiIndex columns
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df["SMA"] = df["Close"].rolling(window=sma_window).mean()
    df["10_SMA"] = df["Close"].rolling(window=10).mean()

    # Compute linear regression slope on last N SMA points
    if len(df.dropna()) < sma_window:
        return None  # not enough data

    slope, _, _, _, _ = linregress(range(smaSlope_window), df["SMA"].tail(smaSlope_window))
    #slope_short, _, _, _, _ = linregress(range(smaSlope_window), df["10_SMA"].tail(smaSlope_window))

    latest_price = df["Close"].iloc[-1]
    latest_sma   = df["SMA"].iloc[-1]
    latest_10sma = df["10_SMA"].iloc[-1]

    # Determine stage
    if (latest_price > latest_sma) and (slope > 0) and (latest_price > latest_10sma) and (latest_10sma > latest_sma):
        stage = "Stage 2 (Advancing)"
    elif latest_price < latest_sma and slope < 0:
        stage = "Stage 4 (Declining)"
    elif np.abs(slope) <= 0.001:
        stage = "Stage 1 (Basing)"
    else:
        stage = "Stage 3 (Topping)"

    return stage, slope, latest_price, latest_sma, latest_10sma


In [13]:

results = []
for etf in etfs:
    df = yf.download(etf, period="3y", interval="1wk", auto_adjust=True)
    stage_info = weinstein_stage(df)
    if stage_info:
        stage, slope, price, sma,sma_10 = stage_info
        results.append({
            "ETF": etf,
            "Stage": stage,
            "SMA_Slope": slope,
            "Latest_Price": price,
            "30W_SMA": sma,
            "10W_SMA": sma_10
        })

stages_df = pd.DataFrame(results).sort_values(by="SMA_Slope", ascending=False)
#print(len(stages_df))
stages_df.head()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,ETF,Stage,SMA_Slope,Latest_Price,30W_SMA,10W_SMA
73,FN,Stage 2 (Advancing),9.521982,722.039978,522.518670,624.041000
12,SITM,Stage 2 (Advancing),7.761950,774.059998,397.902334,498.568005
30,VICR,Stage 2 (Advancing),5.550345,273.670013,154.425000,212.146001
34,BE,Stage 2 (Advancing),4.371804,275.950012,149.615334,200.673000
5,AAOI,Stage 2 (Advancing),3.953147,190.360001,72.861167,138.147000


In [14]:
advancing_stocks= stages_df[stages_df["Stage"] .isin(["Stage 2 (Advancing)"]) ]
advancing_stocks.reset_index(drop=True, inplace=True)
advancing_stocks.head()

,ETF,Stage,SMA_Slope,Latest_Price,30W_SMA,10W_SMA
0,FN,Stage 2 (Advancing),9.521982,722.039978,522.518670,624.041000
1,SITM,Stage 2 (Advancing),7.761950,774.059998,397.902334,498.568005
2,VICR,Stage 2 (Advancing),5.550345,273.670013,154.425000,212.146001
3,BE,Stage 2 (Advancing),4.371804,275.950012,149.615334,200.673000
4,AAOI,Stage 2 (Advancing),3.953147,190.360001,72.861167,138.147000


In [15]:
# List of ETFs to analyze
df_o = df_o[df_o['Asset'].isin(advancing_stocks['ETF'])]
#df_raw = pd.read_csv('etf_list.csv')
#df_raw = df_raw[df_raw['Type'].isin(['ETF','Stock'])]
#recent_quarter = most_recent_quarter_start()
etfs = df_o['Asset'].to_list()
print(etfs)

print(len(etfs))

['RXT', 'MXL', 'DOCN', 'BAND', 'AAOI', 'VPG', 'PENG', 'SATL', 'LUNR', 'SITM', 'AEHR', 'NVTS', 'HUT', 'VSH', 'PLUG', 'AIP', 'SYNA', 'KOPN', 'BKSY', 'CEVA', 'APLD', 'VECO', 'SMTC', 'RIOT', 'VICR', 'PL', 'BE', 'PDFS', 'VIAV', 'TTMI', 'RDW', 'CORZ', 'RMBS', 'VSAT', 'KN', 'LASR', 'CRDO', 'NTCT', 'SANM', 'DGII', 'MTRN', 'COHU', 'NXT', 'SEI', 'PLAB', 'BHE', 'NOVT', 'HLIT', 'MRX', 'NN', 'WULF', 'OPLN', 'ATEN', 'DCO', 'FN']
55


# Brian Shannon daily timeframe stage classification

In [16]:

@dataclass
class ClassifierConfig:
    short_ma:       int   = 20
    medium_ma:      int   = 50
    long_ma:        int   = 200
    atr_period:     int   = 14
    slope_period:   int   = 10       # wider slope window = less noise
    rs_period:      int   = 63       # 3-month RS — more meaningful
    volume_period:  int   = 50       # 50-bar vol average — institutional grade
    pivot_lookback: int   = 10       # wider pivot = fewer false swings

    # Stage 2 thresholds
    stage2_min_score:       int   = 7
    stage2_ma50_slope_min:  float = 0.0
    stage2_ma200_slope_min: float = 0.0    # NEW — 200 MA must also be rising

    # Stage 1 thresholds — tighter = more accurate accumulation ID
    stage1_slope_max:       float = 0.03   # tighter than original 0.05
    stage1_atr_max:         float = 0.04   # tighter volatility compression
    stage1_ma50_proximity:  float = 0.07   # price within 7% of MA50

    # Stage 4 thresholds
    stage4_rs_penalty:      float = -0.05  # RS must be negative for hard Stage 4


# =============================================================
# CLASSIFIER
# =============================================================

class BrianShannonAuctionClassifier:
    """
    Improved Brian Shannon Auction Market Stage Classifier.

    Improvements over original:
    ----------------------------
    1.  Wider slope window (10 bars) — reduces slope noise
    2.  3-month RS period — more institutionally meaningful
    3.  MA200 slope condition added to Stage 2 — prevents false markups
    4.  Stage 1 uses tighter slope + ATR thresholds
    5.  Stage 4 requires negative RS — removes weak/sideways false declines
    6.  Weighted trend score — not all signals are equal
    7.  Stage confidence score added — tells you how strong each call is
    8.  Stage transition detection — flags when stage is changing
    9.  RS percentile rank — tells you where RS stands vs its own history
    10. Multi-stock scanner built in — scan a list and get ranked results
    """

    def __init__(self, config: Optional[ClassifierConfig] = None):
        self.config = config or ClassifierConfig()

    # =========================================================
    # PUBLIC — SINGLE STOCK
    # =========================================================
    def classify(self, df: pd.DataFrame) -> pd.DataFrame:

        df = df.copy()

        self._moving_averages(df)
        self._atr(df)
        self._relative_strength(df)
        self._volume_analysis(df)
        self._trend_structure(df)
        self._trend_score(df)
        self._classify_stages(df)
        self._stage_confidence(df)
        self._stage_transitions(df)

        return df

    # =========================================================
    # PUBLIC — MULTI STOCK SCANNER
    # =========================================================
    def scan(
        self,
        tickers:    list,
        benchmark:  str  = "SPY",
        start:      str  = "2022-01-01",
        min_score:  int  = 0,
        stage_filter: Optional[int] = None
    ) -> pd.DataFrame:
        """
        Scan a list of tickers and return a ranked summary DataFrame.

        Parameters
        ----------
        tickers      : list of ticker symbols
        benchmark    : benchmark ticker for RS (default SPY)
        start        : start date for data download
        min_score    : minimum trend_score to include in results
        stage_filter : filter by stage number (1/2/3/4) or None for all

        Returns
        -------
        pd.DataFrame sorted by trend_score descending
        """

        print(f"\nDownloading benchmark ({benchmark})...")
        spy_data = yf.download(benchmark, start=start, progress=False)

        results = []

        for i, ticker in enumerate(tickers, 1):

            print(f"[{i}/{len(tickers)}] Processing {ticker}...", end=" ")

            try:

                data = yf.download(ticker, start=start, progress=False)

                if data.empty or len(data) < self.config.long_ma + 10:
                    print("SKIP — insufficient data")
                    continue

                # flatten multi-level columns if present
                if isinstance(data.columns, pd.MultiIndex):
                    data.columns = data.columns.get_level_values(0)

                data["benchmark_close"] = spy_data["Close"].reindex(
                    data.index
                ).ffill()

                classified = self.classify(data)
                latest     = classified.iloc[-1]
                prev       = classified.iloc[-2]

                results.append({
                    "Ticker":        ticker,
                    "Close":         round(latest["Close"], 2),
                    "Stage":         int(latest["stage_number"])
                                     if not np.isnan(latest["stage_number"])
                                     else 0,
                    "Stage Label":   latest["stage"],
                    "Trend Score":   int(latest["trend_score"]),
                    "Confidence":    round(latest["stage_confidence"], 1),
                    "RS (3M)":       round(latest["rs"] * 100, 2)
                                     if not np.isnan(latest["rs"]) else np.nan,
                    "RS Percentile": round(latest["rs_percentile"], 1)
                                     if not np.isnan(latest["rs_percentile"])
                                     else np.nan,
                    "Transitioning": latest["stage_transitioning"],
                    "MA20":          round(latest["ma20"], 2),
                    "MA50":          round(latest["ma50"], 2),
                    "MA200":         round(latest["ma200"], 2),
                    "ATR%":          round(latest["atr_pct"] * 100, 2),
                    "Vol Ratio":     round(latest["volume_ratio"], 2),
                    "Accum Days":    int(
                                         classified["accumulation_day"]
                                         .tail(10).sum()
                                     ),
                    "Dist Days":     int(
                                         classified["distribution_day"]
                                         .tail(10).sum()
                                     ),
                })

                print(f"{latest['stage']} | Score: {int(latest['trend_score'])} | Conf: {round(latest['stage_confidence'], 1)}%")

            except Exception as e:
                print(f"ERROR — {e}")
                continue

        if not results:
            print("No results returned.")
            return pd.DataFrame()

        df_results = pd.DataFrame(results)

        # apply filters
        if min_score > 0:
            df_results = df_results[df_results["Trend Score"] >= min_score]

        if stage_filter is not None:
            df_results = df_results[df_results["Stage"] == stage_filter]

        # sort by trend score then confidence
        df_results = df_results.sort_values(
            ["Trend Score", "Confidence"],
            ascending=False
        ).reset_index(drop=True)

        return df_results

    # =========================================================
    # LATEST STAGE — SINGLE STOCK
    # =========================================================
    def latest_stage(self, df: pd.DataFrame) -> dict:

        latest = df.iloc[-1]

        return {
            "date":          latest.name,
            "close":         round(latest["Close"], 2),
            "stage":         latest["stage"],
            "stage_number":  latest["stage_number"],
            "trend_score":   int(latest["trend_score"]),
            "confidence":    round(latest["stage_confidence"], 1),
            "rs_3m":         round(latest["rs"] * 100, 2)
                             if not np.isnan(latest["rs"]) else None,
            "rs_percentile": round(latest["rs_percentile"], 1)
                             if not np.isnan(latest["rs_percentile"]) else None,
            "transitioning": latest["stage_transitioning"],
        }

    # =========================================================
    # MOVING AVERAGES — wider slope window
    # =========================================================
    def _moving_averages(self, df):

        c = self.config

        df["ma20"]  = df["Close"].rolling(c.short_ma).mean()
        df["ma50"]  = df["Close"].rolling(c.medium_ma).mean()
        df["ma200"] = df["Close"].rolling(c.long_ma).mean()

        for ma in ["ma20", "ma50", "ma200"]:
            # normalize slope as % per bar — comparable across price levels
            df[f"{ma}_slope"] = (
                (df[ma] - df[ma].shift(c.slope_period))
                / df[ma].shift(c.slope_period)
            ) / c.slope_period * 100

    # =========================================================
    # ATR
    # =========================================================
    def _atr(self, df):

        hl  = df["High"] - df["Low"]
        hc  = np.abs(df["High"] - df["Close"].shift(1))
        lc  = np.abs(df["Low"]  - df["Close"].shift(1))

        tr       = pd.concat([hl, hc, lc], axis=1).max(axis=1)
        df["ATR"]     = tr.rolling(self.config.atr_period).mean()
        df["atr_pct"] = df["ATR"] / df["Close"]

    # =========================================================
    # RELATIVE STRENGTH — 3 month + percentile rank
    # =========================================================
    def _relative_strength(self, df):

        if "benchmark_close" not in df.columns:
            df["benchmark_close"] = np.nan

        p = self.config.rs_period

        stock_ret     = df["Close"] / df["Close"].shift(p) - 1
        benchmark_ret = df["benchmark_close"] / df["benchmark_close"].shift(p) - 1

        df["rs"] = stock_ret - benchmark_ret

        # RS trend — is RS improving vs its own 20-bar average
        df["rs_trend"] = df["rs"] > df["rs"].rolling(20).mean()

        # RS percentile rank over 1 year — where does current RS sit historically
        df["rs_percentile"] = df["rs"].rolling(252).rank(pct=True) * 100

    # =========================================================
    # VOLUME — 50-bar average, institutional grade
    # =========================================================
    def _volume_analysis(self, df):

        df["avg_volume"]   = df["Volume"].rolling(self.config.volume_period).mean()
        df["volume_ratio"] = df["Volume"] / df["avg_volume"]

        # accumulation day — up on above-average volume
        df["accumulation_day"] = (
            (df["Close"] > df["Close"].shift(1))
            & (df["volume_ratio"] > 1.25)
        )

        # distribution day — down on above-average volume
        df["distribution_day"] = (
            (df["Close"] < df["Close"].shift(1))
            & (df["volume_ratio"] > 1.25)
        )

        # churning — high volume but little price progress (topping signal)
        df["churning"] = (
            (df["volume_ratio"] > 1.5)
            & (np.abs(df["Close"] - df["Close"].shift(1)) / df["Close"] < 0.005)
        )

    # =========================================================
    # TREND STRUCTURE
    # =========================================================
    def _trend_structure(self, df):

        lb = self.config.pivot_lookback

        df["rolling_high"] = df["High"].rolling(lb).max()
        df["rolling_low"]  = df["Low"].rolling(lb).min()

        df["higher_high"]  = df["rolling_high"] > df["rolling_high"].shift(lb)
        df["higher_low"]   = df["rolling_low"]  > df["rolling_low"].shift(lb)
        df["lower_high"]   = df["rolling_high"] < df["rolling_high"].shift(lb)
        df["lower_low"]    = df["rolling_low"]  < df["rolling_low"].shift(lb)

    # =========================================================
    # WEIGHTED TREND SCORE — not all signals equal
    # =========================================================
    def _trend_score(self, df):

        score = np.zeros(len(df))

        # price vs MAs — weight by importance
        score += (df["Close"] > df["ma20"]).astype(int)   * 1
        score += (df["Close"] > df["ma50"]).astype(int)   * 2   # heavier weight
        score += (df["Close"] > df["ma200"]).astype(int)  * 2   # heavier weight

        # full MA alignment — most important single condition
        score += (
            (df["ma20"] > df["ma50"]) & (df["ma50"] > df["ma200"])
        ).astype(int) * 2

        # positive slopes
        score += (df["ma20_slope"]  > 0).astype(int) * 1
        score += (df["ma50_slope"]  > 0).astype(int) * 1
        score += (df["ma200_slope"] > 0).astype(int) * 1        # NEW

        # swing structure
        score += df["higher_high"].astype(int) * 1
        score += df["higher_low"].astype(int)  * 1

        # RS improving AND above benchmark
        score += (
            df["rs_trend"].fillna(False)
            & (df["rs"].fillna(0) > 0)
        ).astype(int) * 1

        # volume confirmation — accumulation days in last 10 bars
        score += (
            df["accumulation_day"].rolling(10).sum() >= 3
        ).astype(int) * 1

        # churning penalty — topping signal
        score -= df["churning"].astype(int) * 1

        df["trend_score"] = score.clip(lower=0)

    # =========================================================
    # STAGE CLASSIFICATION — improved logic
    # =========================================================
    def _classify_stages(self, df):

        c = self.config

        # ── STAGE 2: MARKUP ──────────────────────────────────
        # Requires MA200 slope too — prevents classifying late-stage
        # rallies where 200 MA is still falling as Stage 2
        stage2 = (
            (df["trend_score"] >= c.stage2_min_score)
            & (df["Close"]    > df["ma50"])
            & (df["ma20"]     > df["ma50"])
            & (df["ma50"]     > df["ma200"])
            & (df["ma20_slope"]  > c.stage2_ma50_slope_min)
            & (df["ma50_slope"]  > c.stage2_ma50_slope_min)
            & (df["ma200_slope"] > c.stage2_ma200_slope_min)   # NEW
        )

        # ── STAGE 4: DECLINE ─────────────────────────────────
        # Added RS condition — must be underperforming benchmark
        stage4 = (
            (df["Close"]      < df["ma50"])
            & (df["ma20"]     < df["ma50"])
            & (df["ma50"]     < df["ma200"])
            & (df["ma20_slope"]  < 0)
            & (df["ma50_slope"]  < 0)
            & (df["lower_high"])
            & (df["lower_low"])
            & (df["rs"].fillna(0) < c.stage4_rs_penalty)       # NEW
        )

        # ── STAGE 1: ACCUMULATION ────────────────────────────
        # Tighter thresholds — real accumulation is tight and quiet
        stage1 = (
            (np.abs(df["ma20_slope"])  < c.stage1_slope_max)
            & (np.abs(df["ma50_slope"]) < c.stage1_slope_max)
            & (df["atr_pct"]            < c.stage1_atr_max)
            & (
                np.abs(
                    (df["Close"] - df["ma50"]) / df["ma50"]
                ) < c.stage1_ma50_proximity
            )
            & (~stage2)
            & (~stage4)
        )

        # ── STAGE 3: DISTRIBUTION ────────────────────────────
        # Everything not cleanly Stage 1/2/4
        stage3 = ~(stage1 | stage2 | stage4)

        df["stage"] = np.select(
            [stage1, stage2, stage3, stage4],
            [
                "Stage 1 - Accumulation",
                "Stage 2 - Markup",
                "Stage 3 - Distribution",
                "Stage 4 - Decline",
            ],
            default="Unknown"
        )

        df["stage_number"] = np.select(
            [stage1, stage2, stage3, stage4],
            [1, 2, 3, 4],
            default=np.nan
        )

    # =========================================================
    # STAGE CONFIDENCE — how strongly does price fit the stage
    # =========================================================
    def _stage_confidence(self, df):
      """
       Confidence = how strongly price fits its current stage.
       - Stage 2 (Long):    high score = high confidence
       - Stage 4 (Short):   low score  = high confidence
       - Stage 1/3:       proximity to midpoint 7 = high confidence
      """

      SCORE_MAX = 14.0
      SCORE_MIN = 0.0
      SCORE_MID = 7.0
      ts = df["trend_score"]
      #max_score = df["trend_score"].rolling(252, min_periods=50).max().replace(0, np.nan)
      #min_score = df["trend_score"].rolling(252, min_periods=50).min().replace(0, np.nan)
      #med_score = df["trend_score"].rolling(252, min_periods=50).median().replace(0, np.nan)
      #score_range = (max_score - min_score).replace(0, np.nan)

      # ── Stage 2 confidence — how close to historical peak
      #long_confidence = (df["trend_score"] / max_score * 100).clip(0, 100)
      long_confidence = (ts / SCORE_MAX * 100).clip(0, 100)

      # ── Stage 4 confidence — how close to historical trough
      # invert: low score = high confidence for shorts
      #short_confidence = (1 - (df["trend_score"] - min_score) / score_range ).clip(0, 1) * 100
      short_confidence = ((SCORE_MAX - ts) / SCORE_MAX * 100).clip(0, 100)

      # ── Stage 1/3 confidence — how close to median (sideways)
      #neutral_confidence = (1 - abs(df["trend_score"] - med_score) / score_range).clip(0, 1) * 100
      neutral_confidence = ((1 - abs(ts - SCORE_MID) / SCORE_MID) * 100).clip(0, 100)

      # ── Apply correct confidence per stage
      df["stage_confidence"] = np.select(
        [
            df["stage_number"] == 2,
            df["stage_number"] == 4,
            df["stage_number"].isin([1, 3]),
        ],
        [
            long_confidence,
            short_confidence,
            neutral_confidence,
        ],
        default=50 ).clip(0, 100)

      # fill any NaN with neutral 50
      #df["stage_confidence"] = df["stage_confidence"].fillna(50)

    # =========================================================
    # STAGE TRANSITIONS — detect when stage is changing
    # =========================================================
    def _stage_transitions(self, df):
        """
        Flags bars where stage has changed vs previous bar.
        Useful for catching early stage shifts.
        """

        df["stage_transitioning"] = (
            df["stage_number"] != df["stage_number"].shift(1)
        )

In [17]:
def run_single(ticker="NVDA", benchmark="SPY", start="2022-01-01"):

    #print(f"\n{'='*55}")
    print(f"  SINGLE STOCK ANALYSIS: {ticker}")
    #print(f"{'='*55}")

    data = yf.download(ticker, start=start, progress=False)
    spy  = yf.download(benchmark, start=start, progress=False)

    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)

    data["benchmark_close"] = spy["Close"].reindex(data.index).ffill()

    clf    = BrianShannonAuctionClassifier()
    result = clf.classify(data)

    print(
        result[[
            "Close", "ma20", "ma50", "ma200",
            "trend_score", "stage_confidence", "stage"
        ]].tail(10).to_string()
    )

    print("\nLATEST SIGNAL")
    print("-" * 40)
    latest = clf.latest_stage(result)
    for k, v in latest.items():
        print(f"  {k:<18}: {v}")


In [18]:
# =============================================================
# EXAMPLE — MULTI STOCK SCANNER
# =============================================================

def run_scanner(watchlist=None, stage_filter=2, start="2022-01-01"):

    if watchlist is None:
        print("No watchlist provided. Please pass a list of tickers.")
        return

    clf = BrianShannonAuctionClassifier()

    print(f"\n{'='*55}")
    print(f"  MULTI STOCK SCANNER — {len(watchlist)} tickers")
    print(f"{'='*55}")

    results = clf.scan(
        tickers=watchlist,
        benchmark="SPY",
        start=start,
        stage_filter=stage_filter
    )

    if results.empty:
        print("No stocks matched the filter.")
        return

    stage_label = f"Stage {stage_filter} Only" if stage_filter else "All Stages"

    print(f"\n{'='*55}")
    print(f"  SCAN RESULTS — {stage_label}")
    print(f"{'='*55}\n")

    display_cols = [
        "Ticker", "Close", "Stage Label", "Trend Score",
        "Confidence", "RS (3M)", "RS Percentile",
        "Transitioning", "Accum Days", "Dist Days"
    ]

    #print(results[display_cols].to_string(index=False))
    #print(f"\nTotal matches: {len(results)}")

    return results


In [19]:
# todays list
todays_list = advancing_stocks['ETF'].tolist()
#run_single("NVDA")
# Multi stock scanner — Stage 2 only
results = run_scanner(watchlist=todays_list)

filtered = results[
  # ── Must be Stage 2 — confirmed uptrend
  (results["Stage"] == 2)

  # ── Trend must be strong — not borderline
  & (results["Trend Score"] >= 6)

  # ── High confidence the stage call is correct
  & (results["Confidence"] >= 60)

  # ── RS must be positive — beating the market
  & (results["RS (3M)"] > 0)

  # ── RS percentile — top half of its own history
  & (results["RS Percentile"] >= 50)

  # ── More accumulation than distribution in last 10 days
  & (results["Accum Days"] > results["Dist Days"])

  ].copy()

# ── Rank by composite score: RS Percentile + Confidence + Trend Score
filtered["rank_score"] = (
  filtered["RS Percentile"] * 0.35
  + filtered["Confidence"]  * 0.35
  + filtered["Trend Score"] * 0.30
    )

filtered = filtered.sort_values(
        "rank_score", ascending=False
    ).head(100).reset_index(drop=True)

display_cols = [
        "Ticker", "Close", "Stage Label", "Trend Score",
        "Confidence", "RS (3M)", "RS Percentile",
        "Transitioning", "Accum Days", "Dist Days"
    ]


df_o = df_o[df_o['Asset'].isin(filtered['Ticker'])]
etfs = df_o['Asset'].to_list()
print(etfs)
print(len(etfs))


  MULTI STOCK SCANNER — 55 tickers

[1/55] Processing FN... Stage 2 - Markup | Score: 12 | Conf: 85.7%
[2/55] Processing SITM... Stage 2 - Markup | Score: 14 | Conf: 100.0%
[3/55] Processing VICR... Stage 2 - Markup | Score: 13 | Conf: 92.9%
[4/55] Processing BE... Stage 2 - Markup | Score: 13 | Conf: 92.9%
[5/55] Processing AAOI... Stage 2 - Markup | Score: 14 | Conf: 100.0%
[6/55] Processing TTMI... Stage 2 - Markup | Score: 12 | Conf: 85.7%
[7/55] Processing DOCN... Stage 2 - Markup | Score: 13 | Conf: 92.9%
[8/55] Processing SANM... Stage 2 - Markup | Score: 13 | Conf: 92.9%
[9/55] Processing MTRN... Stage 2 - Markup | Score: 12 | Conf: 85.7%
[10/55] Processing AEHR... Stage 2 - Markup | Score: 12 | Conf: 85.7%
[11/55] Processing NXT... Stage 2 - Markup | Score: 14 | Conf: 100.0%
[12/55] Processing DCO... Stage 2 - Markup | Score: 12 | Conf: 85.7%
[13/55] Processing LASR... Stage 2 - Markup | Score: 12 | Conf: 85.7%
[14/55] Processing SMTC... Stage 2 - Markup | Score: 14 | Conf: 1

In [20]:

def macdv(prices, fast=12, slow=26, signal=9, atr_window=10, thresholds=(50, 150)):
    """
    Compute MACD-V (volatility normalized MACD).

    Parameters
    ----------
    prices : pd.Series
        Price series (e.g. closing prices).
    fast : int
        Fast EMA period.
    slow : int
        Slow EMA period.
    signal : int
        Signal EMA period for MACD line.
    atr_window : int
        ATR lookback window.
    thresholds : tuple
        (lower, upper) thresholds for neutral/ranging and extreme momentum zones.

    Returns
    -------
    pd.DataFrame with columns:
        - MACDV : MACD-V value
        - Signal : EMA of MACDV
        - Histogram : MACDV - Signal
        - EntryFlag : True when momentum is strong enough, False otherwise
    """
    # --- Step 1: EMAs for MACD ---
    ema_fast = prices.ewm(span=fast, adjust=False).mean()
    ema_slow = prices.ewm(span=slow, adjust=False).mean()
    macd_raw = ema_fast - ema_slow

    # --- Step 2: ATR for normalization ---
    high = prices.shift(1) * (1 + 0.01)   # synthetic highs/lows if OHLC not available
    low = prices.shift(1) * (1 - 0.01)
    close = prices
    tr1 = high - low
    tr2 = (high - close.shift(1)).abs()
    tr3 = (low - close.shift(1)).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.rolling(atr_window).mean()

    # --- Step 3: Normalize MACD by ATR ---
    macdv = (macd_raw / atr) * 100

    # --- Step 4: Signal line and histogram ---
    signal_line = macdv.ewm(span=signal, adjust=False).mean()
    histogram = macdv - signal_line

    # --- Step 5: Entry conditions ---
    lower, upper = thresholds
    entry_flag = ((macdv.abs() > lower) & (macdv.abs() < upper)) | (macdv.abs() > upper)

    df = pd.DataFrame({
        "MACDV": macdv,
        "Signal": signal_line,
        "Histogram": histogram,
        "EntryFlag": entry_flag
    })
    curr = df.iloc[-1]

    return curr.EntryFlag

def money_flow_signals(df, period=10):
    """
    Calculate Money Flow Index (MFI) and generate signals:
    - Positive money flow (TP > TP_prev)
    - Divergence (Price vs MFI mismatch)

    Parameters:
        df (pd.DataFrame): DataFrame with columns ["High", "Low", "Close", "Volume"]
        period (int): Lookback period for MFI (default=10)

    Returns:
        pd.DataFrame with added columns: ["TypicalPrice", "MFI", "PositiveFlow", "Divergence"]
    """

    df = df.copy()

    # Step 1: Typical Price
    df["TypicalPrice"] = (df["High"] + df["Low"] + df["Close"]) / 3

    # Step 2: Raw Money Flow
    df["RawMoneyFlow"] = df["TypicalPrice"] * df["Volume"]

    # Step 3: Positive & Negative Flow
    df["PositiveFlow"] = np.where(df["TypicalPrice"] > df["TypicalPrice"].shift(1), df["RawMoneyFlow"], 0.0)
    df["NegativeFlow"] = np.where(df["TypicalPrice"] < df["TypicalPrice"].shift(1), df["RawMoneyFlow"], 0.0)

    # Step 4: Money Flow Ratio & MFI
    pos_flow = df["PositiveFlow"].rolling(period).sum()
    neg_flow = df["NegativeFlow"].rolling(period).sum()
    money_flow_ratio = pos_flow / neg_flow.replace(0, np.nan)
    df["MFI"] = 100 - (100 / (1 + money_flow_ratio))

    # Step 5: Positive Money Flow Signal
    df["PositiveFlowSignal"] = df["TypicalPrice"] > df["TypicalPrice"].shift(1)

    # Step 6: Divergence Detection
    df["PriceHigh"] = df["Close"].rolling(period).max()
    df["PriceLow"] = df["Close"].rolling(period).min()
    df["MFIHigh"] = df["MFI"].rolling(period).max()
    df["MFILow"] = df["MFI"].rolling(period).min()

    def divergence(row):
        if np.isnan(row["MFI"]):
            return None
        # Price makes higher high, but MFI does not
        if row["Close"] >= row["PriceHigh"] and row["MFI"] < row["MFIHigh"]:
            return "Bearish Divergence ⚠️"
        # Price makes lower low, but MFI does not
        elif row["Close"] <= row["PriceLow"] and row["MFI"] > row["MFILow"]:
            return "Bullish Divergence ✅"
        else:
            return "No Divergence"

    df["Divergence"] = df.apply(divergence, axis=1)
    df['Entry_signal']= (df["MFI"] > 50) & (df["Divergence"] == "No Divergence")
    curr = df.iloc[-1]

    return curr.Entry_signal




In [21]:
# Function to fetch historical weekly data
def rolling_regression_slope(series, window=10):
    """Rolling linear regression slope (price units per bar)."""
    def calc_slope(y):
        if len(y) < 2:
            return np.nan
        x = np.arange(len(y))
        slope, _, _, _, _ = linregress(x, y)
        return slope
    return series.rolling(window).apply(calc_slope, raw=False)

def anchored_vwap(ticker: str, lookback_weeks: int = 5):
    """
    Calculate the Anchored VWAP from the most recent high within the past `lookback_weeks`
    and create a 'Signal' column that gives 'Buy' when Close > Anchored_VWAP, else 'No-Buy'.

    Args:
        ticker (str): Stock ticker symbol
        lookback_weeks (int): Number of weeks to look back for the highest price

    Returns:
        pandas.DataFrame: DataFrame with OHLC, Volume, Anchored_VWAP, and Signal columns
    """
    try:
        # --- Fetch 6 months of daily data to cover the lookback window
        data = yf.download(ticker, period="6mo", interval="1d", progress=False,auto_adjust=True)
        if isinstance(data.columns, pd.MultiIndex):
          data.columns = data.columns.get_level_values(0)  # keep only first level

        if data.empty:
            raise ValueError(f"No data retrieved for ticker {ticker}")

        data.dropna(inplace=True)
        data.index = pd.to_datetime(data.index)

        # --- Ensure we have enough data for the lookback period
        min_days_required = lookback_weeks * 10  # ~5 trading days per week
        if len(data) < min_days_required:
            raise ValueError(f"Insufficient data: {len(data)} days available, {min_days_required} required")

        # --- Find the most recent high within the past lookback_weeks
        recent_period = data.tail(min_days_required)
        anchor_date = recent_period['High'].idxmax()

        # --- Verify anchor_date is a valid Timestamp
        if not isinstance(anchor_date, pd.Timestamp):
            raise ValueError(f"Invalid anchor_date: {anchor_date}. Expected a Timestamp.")

        anchor_price = recent_period.loc[anchor_date, 'High']

        # --- Use .date() safely since we confirmed anchor_date is a Timestamp
        print(f"\nAnchored VWAP for {ticker} starting from {anchor_date.date()} (recent high = {anchor_price:.2f})")

        # --- Slice data from the anchor date onwards
        anchor_data = data.loc[anchor_date:]

        # --- Compute VWAP starting from the anchor date
        # Use typical price ((H+L+C)/3) for more accurate VWAP
        typical_price = (anchor_data['High'] + anchor_data['Low'] + anchor_data['Close']) / 3
        q = anchor_data['Volume']
        pv = (typical_price * q).cumsum()
        v = q.cumsum()

        # --- Avoid division by zero
        avwap = pv / v.where(v != 0, np.nan)

        # --- Add Anchored VWAP to the full dataset
        data['Anchored_VWAP'] = np.nan  # Initialize with NaN
        data.loc[anchor_date:, 'Anchored_VWAP'] = avwap
        # --- Add swing high information to the DataFrame
        data['Swing_High_Price'] = anchor_price
        data['Swing_High_Date'] = anchor_date

        # --- Create Buy/No-Buy Signal
        # Only apply signal where Anchored_VWAP is not NaN
        data['Signal'] = np.where(
            (data['Close'] > data['Anchored_VWAP']) & (data['Anchored_VWAP'].notna()),
            True,
            False
        )

        return data[['Anchored_VWAP', 'Signal','Swing_High_Price']]

    except Exception as e:
        print(f"Error processing {ticker}: {str(e)}")
        return None


def anchored_vwap_old(ticker, anchor_date):
    """
    Calculate Anchored VWAP starting from a given anchor_date.
    Works with both single-level and multi-level columns (e.g. yfinance output).
    """
    # --- Step 1: Flatten columns if multi-index ---
    df = yf.download(ticker, period="1y", interval="1d",auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)  # keep only first level

    # --- Step 2: Ensure required columns exist ---
    required_cols = ["High", "Low", "Close", "Volume"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    # --- Step 3: Subset from anchor_date ---
    df_anchor = df.loc[df.index >= pd.to_datetime(anchor_date)].copy()
    if df_anchor.empty:
        raise ValueError(f"No data found on/after {anchor_date}")

    # --- Step 4: Compute typical price ---
    df_anchor["typical_price"] = (df_anchor["High"] + df_anchor["Low"] + df_anchor["Close"]) / 3.0

    # --- Step 5: Cumulative PV and VWAP ---
    df_anchor["cum_pv"] = (df_anchor["typical_price"].astype(float) * df_anchor["Volume"].astype(float)).cumsum()
    df_anchor["cum_vol"] = df_anchor["Volume"].astype(float).cumsum()
    df_anchor["anchored_vwap"] = df_anchor["cum_pv"] / df_anchor["cum_vol"]

    return df_anchor[["anchored_vwap"]]

def get_monthly_data(ticker):
  try:
      df = yf.download(ticker, period="10y", interval="1mo",auto_adjust=True)

      #if isinstance(df.columns, pd.MultiIndex):
        #df.columns = df.columns.get_level_values(0)  # keep only first level
      # Compute MACD using ta
      df['10_month_SMA'] = df['Close'].rolling(window=10).mean()
      df['slope_raw'] = rolling_regression_slope(df['10_month_SMA'], window=5)
      # normalize as % change per month — comparable across all price levels
      df["slope_pct"] = df["slope_raw"] / df["10_month_SMA"] * 100
      # bullish threshold — SMA rising more than 1% per month
      df["slope_bullish"] = df["slope_pct"] > 1.0
      # Calculate MACD and Signal Line
      df['12_EMA'] = df['Close'].ewm(span=12, adjust=False).mean()
      df['26_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
      # Compute MACD Line
      df['MACD_Line'] = df['12_EMA'] - df['26_EMA']
      # Compute Signal Line (9-day EMA of MACD Line)
      df['Signal_Line'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
      df['MACD_Hist'] = df['MACD_Line'] - df['Signal_Line']

      return df
  except Exception as e:
      print("There is an error getting monthly data", e)

def get_weekly_data(ticker):
  try:
      df = yf.download(ticker, period="10y", interval="1wk",auto_adjust=True)
      #if isinstance(df.columns, pd.MultiIndex):
        #df.columns = df.columns.get_level_values(0)  # keep only first level
      df['10_week_SMA'] = df['Close'].rolling(window=10).mean()
      df['30_week_SMA'] = df['Close'].rolling(window=30).mean()
      df['10_EMA'] = df['Close'].ewm(span=10, adjust=False).mean()
      df['20_EMA'] = df['Close'].ewm(span=20, adjust=False).mean()
      # slope on 10 week SMA — short term trend direction
      df["slope_10sma"] = rolling_regression_slope(df["10_week_SMA"], window=10)
      # slope on 30 week SMA — long term trend direction
      df["slope_30sma"] = rolling_regression_slope(df["30_week_SMA"], window=10)
      # normalize both as % per week — comparable across stocks
      df["slope_10sma_pct"] = df["slope_10sma"] / df["10_week_SMA"] * 100
      df["slope_30sma_pct"] = df["slope_30sma"] / df["30_week_SMA"] * 100

      # 10 week SMA rising at least 0.5% per week
      df["slope_10_bullish"] = df["slope_10sma_pct"] > 0.5
      # 30 week SMA rising at least 0.3% per week
      df["slope_30_bullish"] = df["slope_30sma_pct"] > 0.3
      # both rising = confirmed bullish trend
      df["both_slopes_bullish"] = (
              df["slope_10_bullish"] & df["slope_30_bullish"])
      df['30_week_avg_volume'] = df['Volume'].rolling(window=30).mean()
      # Calculate MACD and Signal Line
      df['12_EMA'] = df['Close'].ewm(span=12, adjust=False).mean()
      df['21_EMA'] = df['Close'].ewm(span=21, adjust=False).mean()
      df['26_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
      # Compute MACD Line
      df['MACD_Line'] = df['12_EMA'] - df['26_EMA']
      # Compute Signal Line (9-day EMA of MACD Line)
      df['Signal_Line'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
      df['MACD_Hist'] = df['MACD_Line'] - df['Signal_Line']
      # --- Overhead Resistance Filter ---
      recent_52_weeks = df[-52:]
      max_close_52w = recent_52_weeks['Close'].max().iloc[0]
      max_price = df['Close'].max().iloc[0]
      last_close = df['Close'].iloc[-1].iloc[0]
      # Filter condition: Close is within 15% of 52-week high
      df['No_Overhead_Resistance'] = last_close > (max_close_52w*0.80)
      # --- Above 52 weeks High ---
      df['above_52w_high'] = last_close > max_close_52w
      df['below_52w_high'] = last_close < max_close_52w

      return df
  except Exception as e:
      print("There is an error getting weekly data", e)

def is_macd_bullish(df):
    """
    Determines if there is a bullish signal on the MACD indicator.
    A bullish signal occurs when the MACD line is above the signal line.

    Parameters:
    - df: dataframe. Must have columns 'MACD_Line' and 'Signal_Line'.
    Returns:
    - bool: True if a bullish signal is detected, otherwise False.
    """
    try:
      if df.empty:
        return False
      # Calculate MACD histogram if not already present

      curr = df.iloc[-3:]
      macd_crossover = curr['MACD_Line'].iloc[-1] > curr['Signal_Line'].iloc[-1]
      hist_increasing = (curr['MACD_Hist'].iloc[-1] > curr['MACD_Hist'].iloc[-2]) \
                         or (curr['MACD_Hist'].iloc[-2] > curr['MACD_Hist'].iloc[-3])

      return macd_crossover
              #and (hist_increasing or curr['MACD_Hist'].iloc[-1] > 0)
    except Exception as e:
      print("Something went wrong while computing the MACD", e)

def is_macd_bullish_hr(df):
    """
    Determines if there is a bullish signal on the MACD indicator on hourly chart.
    A bullish signal occurs when the MACD line is above the signal line.

    Parameters:
    - df: dataframe. Must have columns 'MACD_Line' and 'Signal_Line'.
    Returns:
    - bool: True if a bullish signal is detected, otherwise False.
    """
    try:
      if df.empty:
        return False
      # Calculate MACD histogram if not already present

      curr = df.iloc[-3:]
      macd_crossover = curr['MACD_Line'].iloc[-1] > curr['Signal_Line'].iloc[-1]

      return macd_crossover
            # and curr['MACD_Hist'].iloc[-1] > 0
    except Exception as e:
      print("Something went wrong while computing the MACD", e)

def is_macd_bullish_min(df):
    """
    Determines if there is a bullish signal on the MACD indicator on 15 minute chart.
    A bullish signal occurs when the MACD line is above the signal line.

    Parameters:
    - df: dataframe. Must have columns 'MACD_Line' and 'Signal_Line'.
    Returns:
    - bool: True if a bullish signal is detected, otherwise False.
    """
    try:
      if df.empty:
        return False
      # Calculate MACD histogram if not already present

      curr = df.iloc[-3:]
      macd_crossover = curr['MACD_Line'].iloc[-1] > curr['Signal_Line'].iloc[-1]

      return macd_crossover
    except Exception as e:
      print("Something went wrong while computing the MACD", e)

def calculate_bbw(df, window=20):
    """ Calculates Bollinger Bands Width"""
    sma = df['Close'].rolling(window).mean()
    std = df['Close'].rolling(window).std()
    upper = sma + 2*std
    lower = sma - 2*std
    bbw = (upper - lower) / sma
    return bbw

def calculate_vwap(df):
  """ Calculates vwap"""
  try:
    Typical_Price = (df['Close'].values + df['High'].values + df['Low'].values) / 3
    TPV = Typical_Price * df['Volume'].values
    vwap = TPV.cumsum() / df['Volume'].values.cumsum()
    return vwap
  except Exception as e:
    print("Something went wrong while computing the VWAP:", e)
    return None

# Function to compute RSI
def compute_rsi(series, period=10):
  try:
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))
  except Exception as e:
        print("Something went wrong while computing the RSI:", e)
        return None

def calculate_ma(data, length=10, ma_type="WMA"):
    if ma_type == "SMA":
        return data.rolling(window=length).mean()
    elif ma_type == "EMA":
        return data.ewm(span=length, adjust=False).mean()
    elif ma_type == "WMA":
        weights = np.arange(1, length+1)
        return data.rolling(length).apply(lambda x: np.dot(x, weights)/weights.sum(), raw=True)
    elif ma_type == "VWMA":
        return ta.volume_weighted_average_price(data, length)

def calculate_bollinger_bands(data, length=10, std_dev=2.0):
    sma = data.rolling(window=length).mean()
    std_dev = data.rolling(window=length).std()
    upper_band = sma + (std_dev * std_dev)
    lower_band = sma - (std_dev * std_dev)
    return upper_band, lower_band

# Function to compute ATR (Average True Range)
def compute_atr(df, period=10):
  try:
    df['High-Low'] = df['High'] - df['Low']
    df['High-Close'] = abs(df['High'] - df['Close'].shift(1))
    df['Low-Close'] = abs(df['Low'] - df['Close'].shift(1))
    df['TR'] = df[['High-Low', 'High-Close', 'Low-Close']].max(axis=1)
    return df['TR'].rolling(window=period).mean()
  except Exception as e:
      print("Something went wrong whilecomputing the ATR", e)


# Function to compute On-Balance Volume (OBV)
def compute_obv(df):
  try:
    # Calculate daily price change: 1 if price is up, -1 if down, 0 if unchanged
    price_change = df['Close'].diff()

    # Use price change to decide whether to add or subtract volume
    obv = (price_change > 0).astype(int) * df['Volume']  # Volume when price goes up
    obv -= (price_change < 0).astype(int) * df['Volume']  # Volume when price goes down

    # We accumulate the OBV by taking the cumulative sum of the volume changes
    obv = obv.cumsum()

    return obv
  except Exception as e:
      print("Something went wrong while computing the OBV", e)

def is_bullish_engulfing(df):
    prev = df.iloc[-2]
    curr = df.iloc[-1]
    return (
        prev['Close'].iloc[0] < prev['Open'].iloc[0] and # Previous red
        curr['Close'].iloc[0] > curr['Open'].iloc[0] and # Current green
        curr['Close'].iloc[0] > prev['Open'].iloc[0] and
        curr['Open'].iloc[0] < prev['Close'].iloc[0]
    )

# Function to calculate risk-reward ratio
def calculate_risk_reward(df):
  try:
    if df.empty or len(df) < 20:  # Ensure there are enough data points
        return np.nan

    #latest_price = df['Close'].iloc[-1].iloc[0]
    latest_price = df['Close'].iloc[-1]

    # Use the ATR for setting support level
    atr = df['ATR'].iloc[-1]  # Latest ATR value
    price_ema = df['8_day_EMA'].iloc[-1]
    #atr_multiple = 1.5  # You can adjust this multiplier based on your strategy

    # Calculate the support level using the ATR
    trailing = atr
    #stop = price_ema - (trailing * atr_multiple)
    #stop = price_ema + (trailing * atr_multiple)

    return trailing, price_ema
  except Exception as e:
      print("Something went wrong while computing the reward-risk ratio", e)

# Function to fetch daily data
def get_daily_data(ticker):
    df = yf.download(ticker, period="1y", interval="1d",auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)  # keep only first level
    df['20_day_SMA'] = df['Close'].rolling(window=20).mean()
    df['20_day_EMA'] = df['Close'].ewm(span=20, adjust=False).mean()
    df['50_day_avg_volume'] = df['Volume'].rolling(window=50).mean()
    df['8_day_EMA'] = df['Close'].ewm(span=8, adjust=False).mean()
    df['15_day_EMA'] = df['Close'].ewm(span=15, adjust=False).mean()
    df['21_day_EMA'] = df['Close'].ewm(span=21, adjust=False).mean()
    df['26_day_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
    df['5_day_SMA'] = df['Close'].rolling(window=5).mean()
    df['50_day_SMA'] = df['Close'].rolling(window=50).mean()
    df['100_day_SMA'] = df['Close'].rolling(window=100).mean()
    df['200_day_SMA'] = df['Close'].rolling(window=200).mean()
    df['trend_break_up'] = df['Close'] > df['Close'].rolling(5).max().shift(1)
    df['swing_low'] = df['Low'].where(df['trend_break_up']).ffill()
    # Slope for 5-day SMA (short-term trend)
    df['slope5_raw'] = rolling_regression_slope(df['5_day_SMA'], window=10)
    df['slope5_pct_per_day'] = df['slope5_raw'] / df['5_day_SMA']  # fractional change per day
    df['slope5_annualized_pct'] = df['slope5_pct_per_day'] * 252 * 100  # % per year
    # Slope for 50-day SMA (intermediate-term trend)
    df['slope50_raw'] = rolling_regression_slope(df['50_day_SMA'], window=10)
    df['slope50_pct_per_day'] = df['slope50_raw'] / df['50_day_SMA']  # fractional change per day
    df['slope50_annualized_pct'] = df['slope50_pct_per_day'] * 252 * 100  # % per year
    df['ATR'] = compute_atr(df, 10)
    df["8EMA_plus_ATR"] = df["8_day_EMA"] + 0.5* df["ATR"]
    df["8EMA_plus_ATRL"] = df["8_day_EMA"] + 1* df["ATR"]
    # 1️⃣ Yesterday touched or pierced 8 EMA
    df['prior_touch_8ema'] = df['Low'].shift(1) <= df['8_day_EMA'].shift(1)
    # 2️⃣ Today closes above 8 EMA
    df['close_above_8ema'] = df['Close'] > df['8_day_EMA']
    # 3️⃣ Today closes above yesterday’s close (price rising)
    df['close_above_prev_close'] = df['Close'] > df['Close'].shift(1)
    # 4️⃣ Strong confirmation: Break previous high
    df['break_prev_high'] = df['Close'] > df['High'].shift(1)
    # 5️⃣ 8 EMA slope positive (trend filter)
    df['ema8_rising'] = df['8_day_EMA'] > df['8_day_EMA'].shift(1)
    # EMA 8 slope
    df['ema8_slope'] = df['8_day_EMA'] - df['8_day_EMA'].shift(1)
    # EMA 8 slope previous
    df['ema8_slope_prev'] = df['ema8_slope'].shift(1)
    # EMA 8 acceleration
    df['ema8_accel'] = df['ema8_slope'] - df['ema8_slope_prev']
    # EMA 8 slope direction (1 = up, 0 = down)
    df['ema8_dir'] = (df['ema8_slope'] > 0).astype(int)
    # Count direction changes over last 5 days
    df['ema8_direction_changes'] = (
      df['ema8_dir']
      .diff()
      .abs()
      .rolling(5)
      .sum()
      )
    # Chop condition
    df['avoid_chop'] = df['ema8_direction_changes'] >= 3

    # 1. Higher Timeframe Bearish Bias (Monthly + Weekly already confirmed)
    trend_long = df['slope50_raw'] > 0

    # 2. Avoid strong down days and chop
    df['daily_return'] = df['Close'].pct_change()
    avoid_strong_down = df['daily_return'] < -0.035
    avoid_chop = df['ema8_direction_changes'] >= 3


    # --- Common Conditions ---
    df['closed_above_ema8'] = df['Close'] > df['8_day_EMA']
    df['bullish_candle'] = df['Close'] > df['Open']
    df['higher_high'] = df['High'] > df['High'].shift(1)

    # --- Pullback Setups (A / A+) ---
    # 3. EMA 8 Price Action
    df['touched_ema8']      = df['Low'] <= df['8_day_EMA'] * 1.005
    df['strong_lower_wick'] = ((df['Close'] - df['Low']) / (df['High'] - df['Low'] + 0.0001)) > 0.5

    # 4. Advanced Bullish Patterns
    df['bullish_engulfing'] = (
      (df['Close'] > df['Open']) &
      (df['Open'] < df['Close'].shift(1)) &
      (df['Close'] > df['Close'].shift(1))
    )

    df['failed_break_below_intraday'] = (
       (df['Low'] < df['8_day_EMA']) &
       (df['Close'] > df['8_day_EMA'])
    )

    df['failed_break_below_twoday'] = (
       (df['Low'].shift(1) < df['8_day_EMA'].shift(1)) &
       (df['Close'] > df['8_day_EMA'])
    )

    df['failed_break_below'] = (
        df['failed_break_below_intraday'] |
        df['failed_break_below_twoday']
    )

    df['pullback_context'] = df['Close'].shift(1) < df['Close'].shift(2)

    # 5. A and A+ Setups
    df['A_setup_long'] = (
      df['touched_ema8'] &
      df['closed_above_ema8'] &
      df['bullish_candle']
    )
    df['A_setup_long'] = df['A_setup_long'] & (df['Close'] < df['8EMA_plus_ATR'])

    df['A_plus_setup_long'] = (
      df['failed_break_below'] &
      (df['bullish_engulfing'] | df['strong_lower_wick']) &
      df['closed_above_ema8'] &
      (df['Volume'] > df['50_day_avg_volume'])  # optional - institutional validation
    )
    df['A_plus_setup_long'] = df['A_plus_setup_long'] & (df['Close'] < df['8EMA_plus_ATR'])

    # Trend Continuation / Momentum Trades ---
    # NOTE: Entry requires price to be within 1 ATR of 8 EMA (handled upstream)
    df['trend_continuation'] = (
      (df['Close'] >= df['8EMA_plus_ATR']) &              # now explicitly in B+ zone
      (df['Close'] < df['8EMA_plus_ATRL']) &              # optional: cap before extended
      (df['Close'] > df['8_day_EMA']) &                    # Already above EMA
      (df['Close'].shift(1) > df['8_day_EMA'].shift(1)) &  # Was already above yesterday
      df['higher_high'] &                                   # Making higher highs
      (df['ema8_slope'] > 0)                             # EMA sloping up
      & (df['daily_return'] > 0.005)                          # Decent green candle
    )

    df['long_signal'] = (
      trend_long &
      (~avoid_strong_down) &
      (~avoid_chop) &
      (
        df['A_setup_long'] |
        df['A_plus_setup_long'] |
        df['trend_continuation']
      ) &
      df['higher_high']
    )

    # Signal Strength Labeling
    is_pullback = df['A_setup_long'] | df['A_plus_setup_long']
    is_trend = df['trend_continuation']
    df['signal_type'] = 'Generic'
    df.loc[df['long_signal'] & is_pullback & ~is_trend, 'signal_type'] = 'Pullback (A/A+)'
    df.loc[df['long_signal'] & ~is_pullback & is_trend, 'signal_type'] = 'Trend Continuation'
    df.loc[df['long_signal'] & is_pullback & is_trend, 'signal_type'] = 'Hybrid'

    # Strength
    df['signal_strength'] = 'C+'
    df.loc[df['long_signal'] & df['A_plus_setup_long'], 'signal_strength'] = 'A+'
    df.loc[df['long_signal'] & df['A_setup_long'] & ~df['A_plus_setup_long'], 'signal_strength'] = 'A'
    df.loc[df['long_signal'] & df['trend_continuation'], 'signal_strength'] = 'B+'

    # Compute MACD using ta
    df["MACD_Line"] = ta.trend.macd(df["Close"], window_slow=26, window_fast=12)
    df["Signal_Line"] = ta.trend.macd_signal(df["Close"], window_slow=26, window_fast=12, window_sign=9)
    df["MACD_Hist"] = ta.trend.macd_diff(df["Close"], window_slow=26, window_fast=12, window_sign=9)
    df["MACD_Hist_above_zero"] = df["MACD_Hist"] > 0
    df["MACD_Hist_below_zero"] = df["MACD_Hist"] < 0
    df['macd_above_signal'] = df['MACD_Line'] > df['Signal_Line']
    df['macd_below_signal'] = df['MACD_Line'] < df['Signal_Line']
    df['VWAP'] = calculate_vwap(df)
    # Compute raw EFI
    df['EFI'] = (df['Close'].diff()) * df['Volume']
    # Compute EMA of EFI
    df['EFI_EMA'] = df['EFI'].ewm(span=13, adjust=False).mean()
    # Determine if EFI_EMA is rising or falling
    df['EFI_EMA_Trend'] = df['EFI_EMA'].diff().apply(lambda x: 'Rising' if x > 0 else 'Falling')
    # Calculate ADX, +DMI and -DMI
    high = df['High']
    low = df['Low']
    close = df['Close']
    # Calculate directional movements
    up_move = high.diff()
    down_move = -low.diff()
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
    # Calculate True Range (TR)
    tr1 = high - low
    tr2 = (high - close.shift()).abs()
    tr3 = (low - close.shift()).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    # Smooth TR, +DM, and -DM using Wilder’s smoothing
    atr = tr.rolling(window=14).sum()
    plus_dm_series = pd.Series(plus_dm.ravel(), index=df.index).astype(float)
    minus_dm_series = pd.Series(minus_dm.ravel(), index=df.index).astype(float)
    plus_dm_smoothed = plus_dm_series.rolling(window=14).sum()
    minus_dm_smoothed = minus_dm_series.rolling(window=14).sum()
    # Directional Indicators
    plus_di = 100 * (plus_dm_smoothed / atr)
    minus_di = 100 * (minus_dm_smoothed / atr)
    # DX and ADX
    dx = 100 * (np.abs(plus_di - minus_di) / (plus_di + minus_di))
    adx = dx.rolling(window=14).mean()
    # Add results to original DataFrame
    df['+DI'] = plus_di
    df['-DI'] = minus_di
    df['ADX'] = adx
    df['di_flag'] = df['+DI'] > df['-DI']
    df['di_flag'] = df['di_flag'].astype(int)
    df['adx_indicator'] = np.where(df['ADX'].between(14, 60), 1, 0) #np.where(df['ADX'] > 14, 1, 0)
    df['adx_signal'] = df['adx_indicator'] * df['di_flag']

    return df

# Function to fetch hourly data
def get_30min_data(ticker):
    df = yf.download(ticker, interval='30m', period='60d',auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)  # keep only first level

    df['65d_SMA'] = df['Close'].rolling(window=65).mean()
    # Slope calculation: regression over recent 10 bars (~5 trading hours)
    df['slope_raw'] = rolling_regression_slope(df['65d_SMA'], window=10)
    # Normalize → fractional change per 30-minute bar
    df['slope_norm'] = df['slope_raw'] / df['65d_SMA']
    # Annualize to % per year (standard for 30m regular hours)
    bars_per_year = 252 * 13
    df['SMA_Slope'] = df['slope_norm'] * bars_per_year * 100
    # Short-term & medium-term moving averages
    df['21_EMA'] = df['Close'].ewm(span=21, adjust=False).mean()
    df['50_EMA'] = df['Close'].ewm(span=50, adjust=False).mean()
    df['200_EMA'] = df['Close'].ewm(span=200, adjust=False).mean()
    # Calculate MACD and Signal Line
    df['12_EMA'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['26_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
    # Compute MACD Line
    df['MACD_Line'] = df['12_EMA'] - df['26_EMA']
    # Compute Signal Line (9-day EMA of MACD Line)
    df['Signal_Line'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
    df['MACD_Hist'] = df['MACD_Line'] - df['Signal_Line']
    return df


# Function to fetch hourly data
def get_15min_data(ticker):
    df = yf.download(ticker, interval='15m', period='30d',auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)  # keep only first level

    df['130d_SMA'] = df['Close'].rolling(window=130).mean()
    # Slope calculation: regression over recent 10 bars (~5 trading hours)
    df['slope_raw'] = rolling_regression_slope(df['130d_SMA'], window=10)
    # Normalize → fractional change per 15-minute bar
    df['slope_norm'] = df['slope_raw'] / df['130d_SMA']
    # Annualize to % per year (standard for 30m regular hours)
    bars_per_year = 252 * 26
    df['SMA_Slope'] = df['slope_norm'] * bars_per_year * 100
    # Short-term & medium-term moving averages
    df['21_EMA'] = df['Close'].ewm(span=21, adjust=False).mean()
    df['50_EMA'] = df['Close'].ewm(span=50, adjust=False).mean()
    df['200_EMA'] = df['Close'].ewm(span=200, adjust=False).mean()
    return df


# Function to check monthly trend
def is_monthly_trend_bullish(df):
    if df.empty:
        return False


    #adx_ok              = df['adx_signal'].iloc[-1] == 1
    latest_price        = df['Close'].iloc[-1].iloc[0]
    latest_sma          = df['10_month_SMA'].iloc[-1]
    macd_bullish_signal = df['MACD_Line'].iloc[-1] > df['Signal_Line'].iloc[-1]
    slope_signal        = df["slope_bullish"].iloc[-1]
    above_10_month_SMA  = (latest_price > latest_sma)

    return above_10_month_SMA and slope_signal and macd_bullish_signal


# Function to check weekly trend
def is_weekly_trend_bullish(df):
    if df.empty:
        return False

    latest_price = df['Close'].iloc[-1].iloc[0]
    latest_30sma = df['30_week_SMA'].iloc[-1]
    latest_10sma = df['10_week_SMA'].iloc[-1]
    sma_10_above_30 = latest_10sma > latest_30sma
    above_10_week_SMA = latest_price > latest_10sma
    above_30_week_SMA = latest_price > latest_30sma
    sma_slope = df["both_slopes_bullish"].iloc[-1]
    macd_bullish_signal =  df['MACD_Line'].iloc[-1] > df['Signal_Line'].iloc[-1]
    trend_ok = sma_10_above_30  and above_10_week_SMA and above_30_week_SMA and sma_slope
    no_overhead_supply = df['No_Overhead_Resistance'].iloc[-1]
    above_52w_high = df['above_52w_high'].iloc[-1]
    time.sleep(2)  # Add a delay of 1 second between requests

    return  trend_ok  and macd_bullish_signal



# Function to check daily entry signal
def is_daily_entry_signal(df2):
    if df2.empty:
        return False

    df = df2.copy()
    counter_trend_long_signal = df['long_signal'].iloc[-1]
    latest_price = df['Close'].iloc[-1]
    sma_slope_5  = df['slope5_annualized_pct'].iloc[-1] > 10
    sma_slope_50 = df['slope50_annualized_pct'].iloc[-1] > 18
    latest_8ema = df['8_day_EMA'].iloc[-1]
    latest_5sma = df['5_day_SMA'].iloc[-1]
    latest_15ema = df['15_day_EMA'].iloc[-1]
    latest_20sma = df['20_day_SMA'].iloc[-1]
    latest_50sma = df['50_day_SMA'].iloc[-1]
    latest_100sma = df['100_day_SMA'].iloc[-1]
    latest_200sma = df['200_day_SMA'].iloc[-1]
    above_20sma = latest_price > latest_20sma
    above_50sma = latest_price > latest_50sma
    above_100sma = latest_price > latest_100sma
    above_200sma = latest_price > latest_200sma
    above_8ema = latest_price >= latest_5sma
    is_8ema_above_15ema = latest_8ema > latest_15ema
    is_20sma_above_50sma = latest_20sma > latest_50sma
    is_50sma_above_100sma = latest_50sma > latest_100sma
    is_50sma_above_200sma = latest_50sma > latest_200sma
    is_100sma_above_200sma = latest_100sma > latest_200sma
    volume_ok = df['Volume'].iloc[-1] >= 1.* df['50_day_avg_volume'].iloc[-1] # Institutional interest
    volume_ok = volume_ok
    macd_level_above_signal = df['macd_above_signal'].iloc[-1]
    macd_bullish_signal = df["MACD_Hist_above_zero"].iloc[-1] and macd_level_above_signal
    #vwap_price = df['VWAP'].iloc[-1]
    elderforce_trend_ok = df['EFI_EMA_Trend'].iloc[-1] == 'Rising'
    elderforce_ema_ok = df['EFI_EMA'].iloc[-1] > 0
    adx_ok = df['adx_signal'].iloc[-1] == 1
    slopes_ok =  sma_slope_5 and sma_slope_50 and volume_ok
    moving_averages_ok = above_50sma and above_100sma and above_200sma \
                          and is_50sma_above_100sma \
                          and is_50sma_above_200sma and is_100sma_above_200sma \

    # Look for a breakout above 20-day SMA & RSI > 50
    return moving_averages_ok and slopes_ok and adx_ok and macd_bullish_signal and elderforce_ema_ok


def get_heikin_ashi_signal(ticker="AAPL", period="6mo", interval="1d"):
    # Fetch OHLC data
    df = yf.download(ticker, period=period, interval=interval, auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)  # keep only first level

    # Compute Heikin Ashi candles
    ha_df = pd.DataFrame(index=df.index)
    ha_df['HA_Close'] = (df['Open'] + df['High'] + df['Low'] + df['Close']) / 4

    ha_open = []
    for i in range(len(df)):
        if i == 0:
             ha_open.append((df['Open'].iloc[i] + df['Close'].iloc[i]) / 2)
        else:
            ha_open.append((ha_open[i-1] + ha_df['HA_Close'].iloc[i-1]) / 2)
    ha_df['HA_Open'] = ha_open
    ha_df['HA_High'] = ha_df[['HA_Open', 'HA_Close']].assign(High=df['High']).max(axis=1)
    ha_df['HA_Low'] = ha_df[['HA_Open', 'HA_Close']].assign(Low=df['Low']).min(axis=1)

    # Combine with original
    df = df.join(ha_df)
    # Check for green candle with flat bottom
    last = df.iloc[-1]
    green_candle = last['HA_Close'] > last['HA_Open']
    flat_bottom = abs(last['HA_Open'] - last['HA_Low']) < 0.01  # tiny wick or flat bottom tolerance
    signal = green_candle and flat_bottom

    print(f"\n🔍 Checking {ticker} ({interval} timeframe)")
    print(f"HA_Open: {last['HA_Open']:.2f}, HA_Close: {last['HA_Close']:.2f}, HA_Low: {last['HA_Low']:.2f}")
    if signal:
        print("✅ Heikin Ashi candle is GREEN with a FLAT bottom — strong bullish signal!")
    elif green_candle:
        print("🟢 Candle is green but not flat-bottomed — still bullish, but less strong.")
    else:
        print("🔴 Not a bullish candle — no entry confirmation yet.")

    return signal, green_candle

# Check entry conditions
def check_entry_conditions(tickers):
    results = []
    for ticker in tickers:
      df = get_daily_data(ticker)
      latest_price = df['Close'].iloc[-1]
      prev_price = df['Close'].iloc[-2]
      latest_sma = df['50_day_SMA'].iloc[-1]
      latest_price_8ema =df['8_day_EMA'].iloc[-1]
      latest_price_21ema =df['21_day_EMA'].iloc[-1]
      price_threshold_ATR = df['8EMA_plus_ATR'].iloc[-1]
      price_threshold_ATRL = df['8EMA_plus_ATRL'].iloc[-1]
      macd_bullish_signal = df['macd_above_signal'].iloc[-1]
      signal_strength      = df['signal_strength'].iloc[-1]
      mfi_signal = money_flow_signals(df)
      # Print results
      print(f"\nMoney flow indicator for {ticker} is:")
      print(mfi_signal)
      macdv_signal = macdv(df['Close'])
      print(f"\nMacd-V indicator for {ticker} is:")
      print(macdv_signal )


      df_entry              = get_30min_data(ticker)
      latest_priceh_5sma    = df_entry['65d_SMA'].iloc[-1]
      latest_priceh         = df_entry['Close'].iloc[-1]
      priceh_buy            = latest_priceh > latest_priceh_5sma
      slope_hr              = df_entry['SMA_Slope'].iloc[-1]> 5
      HA_buy_signal_h,gc_h  = get_heikin_ashi_signal(ticker, period="90d", interval="1d")

      df_refined_entry      = get_15min_data(ticker)
      latest_pricem_5sma    = df_refined_entry['130d_SMA'].iloc[-1]
      latest_pricem         = df_refined_entry['Close'].iloc[-1]
      pricem_buy            = latest_pricem > latest_pricem_5sma
      slope_m               = df_refined_entry['SMA_Slope'].iloc[-1]> 5

      prior_touch_ema       = df['prior_touch_8ema'].iloc[-1]
      close_above_8ema      = df['close_above_8ema'].iloc[-1]
      close_above_prev_close= df['close_above_prev_close'].iloc[-1]
      break_prev_high       = df['break_prev_high'].iloc[-1]
      ema8_rising           = df['ema8_rising'].iloc[-1]
      #aline_refinement      = prior_touch_ema and close_above_8ema and close_above_prev_close and (break_prev_high or ema8_rising)


      # Strict entry — flat top red HA required
      refined_entry_signal_strict = (
          slope_hr and slope_m and
          priceh_buy and pricem_buy and
          HA_buy_signal_h       )

      # Standard entry — just red HA candle required
      refined_entry_signal_standard = (
          slope_hr and slope_m and
          priceh_buy and pricem_buy and
          gc_h   )

      #refined_entry_signal =  sma_slope_h or sma_slope_m
      # Use strict for A+ signals, standard for A/B+
      if signal_strength == 'A+' :
        refined_entry_signal = refined_entry_signal_strict
      else:
        refined_entry_signal = refined_entry_signal_standard


      if latest_price > price_threshold_ATRL:
        entry_signal = "Extended Momentum Entry"

      elif latest_price >= price_threshold_ATR :
        if refined_entry_signal:
           entry_signal = "True Trend Entry"
        else:
           entry_signal = "Other"
      elif  latest_price >= latest_price_8ema and (gc_h or HA_buy_signal_h ):
          entry_signal = "Aline Entry"
      else:
        entry_signal = "Other"
      results.append([ticker, entry_signal])
    # Convert results to DataFrame
    df_results = pd.DataFrame(results, columns=["Asset", "Entry_Signal"])
    return df_results

# Multi-timeframe strategy check returning a DataFrame
def check_mtf_entry(tickers):
    results = []

    for ticker in tickers:
        monthly_df = get_monthly_data(ticker)
        weekly_df = get_weekly_data(ticker)
        daily_df = get_daily_data(ticker)

        if is_monthly_trend_bullish(monthly_df):
            if is_weekly_trend_bullish(weekly_df):
                entry_signal = "Entry Confirmed ✅"
                results.append([ticker, entry_signal])
            else:
                entry_signal = "No Entry Yet on Weekly Timeframe ⏳"
                #results.append([ticker, entry_signal])
        else:
            entry_signal = "Monthly Trend is Not Bullish ❌"
            #results.append([ticker, entry_signal])

        #results.append([ticker, entry_signal])
        time.sleep(2)  # Add a delay of 1 second between requests

    # Convert results to DataFrame
    df_results = pd.DataFrame(results, columns=["Asset", "Entry_Signal"])
    return df_results



In [22]:
# Multi-time frame entry Check
etfs_to_check = etfs #df_o['Asset'].tolist()
# for quick testing
#etfs_to_check  =['EZA', 'GM', 'MU', 'LRCX', 'NVDA', 'CAT', 'WDC','ILF']
df_signals = check_mtf_entry(etfs_to_check)

df_signals

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,Asset,Entry_Signal
0,RXT,Entry Confirmed ✅
1,MXL,Entry Confirmed ✅
2,DOCN,Entry Confirmed ✅
3,BAND,Entry Confirmed ✅
4,AAOI,Entry Confirmed ✅
5,VPG,Entry Confirmed ✅
6,PENG,Entry Confirmed ✅
7,SATL,Entry Confirmed ✅
8,LUNR,Entry Confirmed ✅
9,SITM,Entry Confirmed ✅


## Generate buy list

In [23]:
df_final = df_signals[df_signals['Entry_Signal'] =="Entry Confirmed ✅"]
final_etfs_to_check = df_final['Asset'].tolist()
to_remove = ['PX', 'ABC', 'TLT']

final_etfs_to_check = [x for x in final_etfs_to_check if x not in to_remove]

buy_list = check_entry_conditions(final_etfs_to_check)

buy_list.head()


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Money flow indicator for RXT is:
True

Macd-V indicator for RXT is:
True



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed



🔍 Checking RXT (1d timeframe)
HA_Open: 6.01, HA_Close: 6.42, HA_Low: 5.79
🟢 Candle is green but not flat-bottomed — still bullish, but less strong.


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

...

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Money flow indicator for FN is:
False

Macd-V indicator for FN is:
True



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed



🔍 Checking FN (1d timeframe)
HA_Open: 687.96, HA_Close: 717.57, HA_Low: 687.96
✅ Heikin Ashi candle is GREEN with a FLAT bottom — strong bullish signal!


,Asset,Entry_Signal
0,RXT,Aline Entry
1,MXL,Other
2,DOCN,Other
3,BAND,Extended Momentum Entry
4,AAOI,Other


# Find and filter correlated assets to reduce concentration risk.

In [24]:
def check_abnormal_selling(df, lookback=10, volume_threshold=2.5, price_threshold=-0.02):
    """
    Checks last `lookback` days for statistically abnormal selling activity.
    Flags if any single day shows both volume spike AND strong negative close.
    """
    recent = df.iloc[-lookback:].copy()

    # Volume z-score over last 50 days
    vol_mean = df['Volume'].iloc[-60:-10].mean()
    vol_std  = df['Volume'].iloc[-60:-10].std()
    recent['volume_zscore'] = (recent['Volume'] - vol_mean) / vol_std

    # Flag days with abnormal volume AND strong negative close
    recent['abnormal_selling'] = (
        (recent['volume_zscore'] > volume_threshold) &    # volume > 2.5 std devs
        (recent['daily_return'] < price_threshold)         # strong down day > 2%
    )

    # Check if any such day exists in lookback window
    abnormal_selling_detected = recent['abnormal_selling'].any()

    # How recent is it
    if abnormal_selling_detected:
        days_since = lookback - recent['abnormal_selling'].values[::-1].argmax()
        high_risk = days_since <= 2   # within last 2 days is most dangerous
    else:
        days_since = None
        high_risk = False

    return {
        'abnormal_selling_detected': abnormal_selling_detected,
        'days_since': days_since,
        'high_risk': high_risk
    }


def get_uncorrelated_picks(tickers, ranked_picks, threshold=0.66, period="3mo", interval="1d"):
    """
    Filters a ranked list of tickers to return only uncorrelated picks.

    Parameters:
    -----------
    tickers : list
        All candidate tickers.
    ranked_picks : list
        Ranked list of tickers (best to worst).
    threshold : float
        Correlation threshold (default 0.66).
    period : str
        Data period for yfinance (default "3mo").
    interval : str
        Data interval (default "1d").

    Returns:
    --------
    final_selection : list
        List of uncorrelated tickers.
    corr_matrix : DataFrame
        Correlation matrix of daily returns.
    """
    # Step 1: Get prices
    data = yf.download(tickers, period=period, interval=interval,auto_adjust=True)["Close"]
    data = data.ffill()

    # Step 2: Convert to daily returns
    returns = data.pct_change().dropna()

    # Step 3: Correlation matrix
    corr_matrix = returns.corr()

    # Step 4: Filter uncorrelated picks
    final_selection = []
    for pick in ranked_picks:
        if all(abs(corr_matrix.loc[pick, sel]) <= threshold for sel in final_selection):
            final_selection.append(pick)

    return final_selection, corr_matrix



In [25]:
# Apply TA filters and prioritize ETFs
results = []
buy_list = buy_list[buy_list['Entry_Signal'].isin([
    'Aline Entry',
    'True Trend Entry' ,
    'Extended Momentum Entry'
])]
quad_witching_date = get_last_quad_witching()  # auto-detects most recent

for etf in buy_list['Asset'].to_list():
   df          = get_daily_data(etf)
   price       = df['Close'].iloc[-1]
   selling_check = check_abnormal_selling(df)
   abnormal_selling_check = selling_check['high_risk']
   swing_low   = df['swing_low'].iloc[-1]
   above_21EMA = price > df['21_day_EMA'].iloc[-1]
   above_50sma = price  > df['50_day_SMA'].iloc[-1]

   sma_slope_50 = df['slope50_annualized_pct'].iloc[-1]> 0
   vwap_df     = anchored_vwap(etf, lookback_weeks=2)
   vwap_sy     = anchored_vwap_old(etf, start_of_year)
   vwap_qw     = anchored_vwap_old(etf, quad_witching_date)
   ytd_vwap    = vwap_sy['anchored_vwap'].iloc[-1]
   qw_vwap    = vwap_qw['anchored_vwap'].iloc[-1]


   # WTD
   #vwap_wtd     = anchored_vwap_old(etf, first_day_week )
   #wtd_vwap    = vwap_wtd['anchored_vwap'].iloc[-1]
   print("Current price is :", price)
   print("Most recent quad witching AVWAP is :", qw_vwap)
   print("Year to date VWAP is :", ytd_vwap)
   signal_strength = df['signal_strength'].iloc[-1]
   print("Signal Strength is :", signal_strength)
   #signal_filter = (signal_strength == 'A+' or signal_strength == 'A' or signal_strength == 'B+')


   signal_type = df['signal_type'].iloc[-1]
   print("Signal Type is :", signal_type)


   vwap        = vwap_df['Anchored_VWAP'].iloc[-1]
   vwap_signal = vwap_df['Signal'].iloc[-1]

   swing_high =  vwap_df['Swing_High_Price'].iloc[-1]
   above_vwap  = price > vwap
   above_ytd_vwap = price > ytd_vwap
   above_qw_vwap = price > qw_vwap
   atr_multiple_map = {
     'A+': 1.5,
     'A':  1.5,
     'B+': 2.0,
     'C+': 2.0 }
   atr_multiple = atr_multiple_map.get(signal_strength, 1.5)  # default 1.5 if not found
   # Only use swing low if it's within last 10 bars, otherwise fall back to ATR stop
   swing_low_age = df['trend_break_up'].iloc[-10:].any()

   rr_map = {
     'A+': 3.0,
     'A':  2.5,
     'B+': 2.0,
     'C+': 1.5 }

   rr_multiple = rr_map.get(signal_strength, 1.5)

   entry_buffer_map = {
     'A+': 0.05,
     'A':  0.05,
     'B+': 0.10,
     'C+': 0.10 }


   if  (sma_slope_50 and vwap_signal and above_qw_vwap and not abnormal_selling_check and above_ytd_vwap)  :
    trail, price_ema = calculate_risk_reward(df)
    entry_buffer     = min(0.25, entry_buffer_map.get(signal_strength, 0.10) * trail)
    entry_price = price + entry_buffer #  Move stop to breakeven after 1R achieved on any signal
    stop = price_ema - (atr_multiple * trail)
    if swing_low_age:
        support_level= np.minimum(stop, swing_low - 0.5*trail)
    else:
        support_level = stop

    risk = np.abs(entry_price- support_level)
    breakeven_trigger = entry_price + (1.0 * risk)

    if signal_strength == 'A+':
      take_profit_1=  entry_price+ (2.0 *risk)
      resistance_level = entry_price + (3.0 *risk)
      trail_method     = '8_EMA'
      #time_stop        = 'None'

    elif signal_strength == 'A':
      take_profit_1    =  entry_price+ (1.5 *risk)
      resistance_level = entry_price + (2.5 *risk)
      trail_method     = 'None'
      #time_stop        = 'None'

    elif signal_strength == 'B+':
      take_profit_1=  entry_price+ (2.0 *risk)
      resistance_level = entry_price + (2.0 *risk)
      #time_stop        = 5 # exit after 5 bars if target not hit
      trail_method     = 'None'

    elif signal_strength == 'C+':
      take_profit_1=  entry_price+ (1.5*risk)
      resistance_level = entry_price + (1.5 *risk)
      #time_stop        = 3 # exit after 3 bars if target not hit
      trail_method     = 'None'

    reward = resistance_level - entry_price
    risk_reward_ratio = reward / risk
    # Ensure risk is greater than zero before division
    if risk > 0:
        rr_ratio_1  = np.abs(take_profit_1- entry_price) / risk
        rr_ratio  = reward / risk
    else:
        rr_ratio = np.nan

    take_profit1_perc = ((take_profit_1- entry_price )/entry_price )*100
    stop_loss_perc = ((support_level- entry_price)/entry_price )*100
    take_profit_perc = ((resistance_level- entry_price )/entry_price )*100

    # Fetch the Entry_Signal from buy_list
    entry_signal = buy_list.loc[buy_list['Asset'] == etf, 'Entry_Signal'].values[0]

    # Append results with Entry_Signal
    results.append({
            "Asset": etf,
            "breakeven": breakeven_trigger,
            "Risk-Reward": rr_ratio,
            "Stop Out Price": support_level,
            "Take Profit1": take_profit_1,
            "Target Price": resistance_level,
            #"Current Price": price,
            "Entry Price": entry_price,
            "Trail Price": trail,
            "Entry Signal": entry_signal,  # Add entry signal
            "stop_loss_perc": stop_loss_perc,
            #"take_profit1_perc": take_profit1_perc,
            #"take_profit_perc": take_profit_perc,
            "Anchored VWAP": vwap,
            "signal_type": signal_type,
            "signal_strength": signal_strength
            #"MTD VWAP": mtd_vwap
            #"WTD VWAP": wtd_vwap,
            #"YTD VWAP": ytd_vwap
        })

    time.sleep(2)  # Add a delay of 1 second between requests


# Sort ETFs by highest risk-to-reward ratio
try:
   df_results = pd.DataFrame(results).dropna().sort_values(by="Risk-Reward", ascending=False).reset_index(drop = True)
except Exception as e:
  print("No Asset to buy today, check back some other time!")
  df_results = pd.DataFrame({"Asset": ["No Asset available"]})

df2 = df_results.merge(df_o[['Asset','Type', 'score']], on='Asset', how='left')
df2['timestamp'] = datetime.now()
df2 = df2.sort_values(by='score', ascending=False)

df2.head()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Anchored VWAP for RXT starting from 2026-05-14 (recent high = 7.65)
Current price is : 5.820000171661377
Most recent quad witching AVWAP is : 3.9581814030104607
Year to date VWAP is : 2.4616966657451496
Signal Strength is : C+
Signal Type is : Generic



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Anchored VWAP for BAND starting from 2026-05-15 (recent high = 56.37)
Current price is : 53.970001220703125
Most recent quad witching AVWAP is : 35.634405645010325
Year to date VWAP is : 28.068552524853846
Signal Strength is : C+
Signal Type is : Generic



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

...


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Current price is : 27.950000762939453
Most recent quad witching AVWAP is : 25.349281316449424
Year to date VWAP is : 22.41731287676004
Signal Strength is : A
Signal Type is : Pullback (A/A+)


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Anchored VWAP for FN starting from 2026-05-14 (recent high = 748.89)
Current price is : 722.0399780273438
Most recent quad witching AVWAP is : 636.6081795049454
Year to date VWAP is : 562.2969317783967
Signal Strength is : C+
Signal Type is : Generic


,Asset,breakeven,Risk-Reward,Stop Out Price,Take Profit1,Target Price,Entry Price,Trail Price,Entry Signal,stop_loss_perc,Anchored VWAP,signal_type,signal_strength,Type,score,timestamp
1,BAND,66.629767,1.5,41.810235,72.834650,72.834650,54.220001,3.758000,Extended Momentum Entry,-22.887801,53.823334,Generic,C+,Stock,4.35,2026-05-17 16:32:59.802072
2,SATL,13.641586,1.5,6.239714,15.492054,15.492054,9.940650,1.006500,Extended Momentum Entry,-37.230321,9.346667,Generic,C+,Stock,3.42,2026-05-17 16:32:59.802072
0,FN,864.081209,1.5,580.498747,934.976825,934.976825,722.289978,54.576007,True Trend Entry,-19.630790,721.558907,Generic,C+,Stock,0.34,2026-05-17 16:32:59.802072


## Sentiment Score

In [26]:
NEWS_API_KEY = "15c99612003d4971ad86698b50ed0bd7"  # Get one free from https://newsapi.org/
LOOKBACK_DAYS = 3

# Fetch recent news
def fetch_news(ticker, lookback_days=3):
    url = f"https://newsapi.org/v2/everything?q={ticker}&language=en&from={(datetime.now() - timedelta(days=lookback_days)).date()}&apiKey={NEWS_API_KEY}"
    resp = requests.get(url).json()
    if "articles" not in resp:
        return []
    return [a["title"] for a in resp["articles"]]

# Finbert Sentiment Scoring
finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert")

def get_sentiment_scores(news_list):
    if not news_list:
        return 0
    results = finbert(news_list)
    time.sleep(2)  # Add a delay of 1 second between requests
    scores = [1 if r["label"] == "positive" else -1 if r["label"] == "negative" else 0 for r in results]
    return np.mean(scores)

def build_sentiment_table(TICKERS):
    records = []
    for ticker in TICKERS:
        print(f"Processing {ticker}...")
        news = fetch_news(ticker, LOOKBACK_DAYS)
        sentiment_score = get_sentiment_scores(news)
        combined = {
            "Ticker": ticker,
            "Sentiment": sentiment_score
        }
        records.append(combined)
    df = pd.DataFrame(records)

    # Weighted score (adjustable)
    df["Composite_Score"] = (

        df["Sentiment"].rank(pct=True)
    )

    df = df.sort_values("Composite_Score", ascending=False).reset_index(drop=True)
    return df

# Run sentiment scoring
tickers = df2['Asset'].tolist()
results = build_sentiment_table(tickers)
top_assets = results[results["Sentiment"] >= 0]

#top_assets.head()
top_assets

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Processing BAND...
Processing SATL...
Processing FN...


,Ticker,Sentiment,Composite_Score
0,FN,0.029412,1.000000
1,BAND,0.020202,0.666667
2,SATL,0.000000,0.333333


# US Stock Entries (Day Trade)

In [27]:
# SP 500 stocks
# Fetch the Entry_Signal from buy_list
try:
  df3 = df2[df2['Asset'].isin(top_assets['Ticker'])]
  sp500_stocks_dt = df3[(df3['Type'] == 'Stock') & (df3['Entry Signal'] == 'Extended Momentum Entry')].reset_index(drop=True)
except Exception as e:
  print("No Asset to buy today, check back some other time!")
  sp500_stocks_dt = pd.DataFrame({"Asset": ["No Asset available"]})

sp500_stocks_dt




,Asset,breakeven,Risk-Reward,Stop Out Price,Take Profit1,Target Price,Entry Price,Trail Price,Entry Signal,stop_loss_perc,Anchored VWAP,signal_type,signal_strength,Type,score,timestamp
0,BAND,66.629767,1.5,41.810235,72.834650,72.834650,54.220001,3.7580,Extended Momentum Entry,-22.887801,53.823334,Generic,C+,Stock,4.35,2026-05-17 16:32:59.802072
1,SATL,13.641586,1.5,6.239714,15.492054,15.492054,9.940650,1.0065,Extended Momentum Entry,-37.230321,9.346667,Generic,C+,Stock,3.42,2026-05-17 16:32:59.802072


# US Stock Entries (Aline Entry)

In [28]:
# Fetch the Entry_Signal from buy_list
# Filter US stocks for Aline Entry
try:
  us_stocks = df3[(df3['Type'] == 'Stock') & (df3['Entry Signal'].isin(['Aline Entry']))].reset_index(drop=True)

  tickers = us_stocks['Asset'].tolist()  # Replace with your list of tickers
  ranked_picks = tickers

  final_stock_selection, corr_matrix = get_uncorrelated_picks(tickers, ranked_picks, threshold=0.66)

  #print("Final uncorrelated picks:", final_selection)
  print("\nCorrelation matrix:\n", corr_matrix)

  # Keep only rows where Asset is in filtered
  filtered_stock_list = us_stocks [us_stocks ["Asset"].isin(final_stock_selection)]
except Exception as e:
  print("No Asset to buy today, check back some other time!")
  filtered_stock_list = pd.DataFrame({"Asset": ["No Asset available"]})

filtered_stock_list

[*********************100%***********************]  0 of 0 completed

No Asset to buy today, check back some other time!


,Asset
0,No Asset available


# US Stock Entries (True Trend Entry)

In [30]:
# Fetch the Entry_Signal from buy_list
# Filter US stocks for Aline Entry
try:
  us_stocks = df3[(df3['Type'] == 'Stock') & (df3['Entry Signal'].isin(['True Trend Entry']))].reset_index(drop=True)

  tickers = us_stocks['Asset'].tolist()  # Replace with your list of tickers
  ranked_picks = tickers
  if len(ranked_picks) > 1:
    final_stock_selection, corr_matrix = get_uncorrelated_picks(tickers, ranked_picks, threshold=0.66)
    print("\nCorrelation matrix:\n", corr_matrix)
  else:
    final_stock_selection = ranked_picks


  # Keep only rows where Asset is in filtered
  filtered_stock_list2 = us_stocks [us_stocks ["Asset"].isin(final_stock_selection)]
except Exception as e:
  print("No Asset to buy today, check back some other time!")
  filtered_stock_list2 = pd.DataFrame({"Asset": ["No Asset available"]})

filtered_stock_list2
#final_stock_selection

,Asset,breakeven,Risk-Reward,Stop Out Price,Take Profit1,Target Price,Entry Price,Trail Price,Entry Signal,stop_loss_perc,Anchored VWAP,signal_type,signal_strength,Type,score,timestamp
0,FN,864.081209,1.5,580.498747,934.976825,934.976825,722.289978,54.576007,True Trend Entry,-19.63079,721.558907,Generic,C+,Stock,0.34,2026-05-17 16:32:59.802072
